In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sksurv.metrics import concordance_index_ipcw, concordance_index_censored
from sksurv.util import Surv
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

class TimeDiscretizer:
    def __init__(self, num_bins=50):
        self.num_bins = num_bins
        self.cuts = None
        
    def fit(self, times, events):
        uncensored = times[events == 1]
        try:
            _, self.cuts = pd.qcut(uncensored, q=self.num_bins, retbins=True, duplicates='drop')
        except:
            self.cuts = np.linspace(times.min(), times.max(), self.num_bins + 1)
        self.cuts[0] = 0
        self.cuts[-1] = max(times.max(), self.cuts[-1]) + 1e-5
        self.num_bins = len(self.cuts) - 1
        
    def transform(self, times):
        return np.clip(np.digitize(times, self.cuts) - 1, 0, self.num_bins - 1)


class SurvivalDataset(Dataset):
    def __init__(self, X, time_indices, events, durations):
        self.X = torch.FloatTensor(X)
        self.time_indices = torch.LongTensor(time_indices)
        self.events = torch.FloatTensor(events)
        self.durations = torch.FloatTensor(durations)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.time_indices[idx], self.events[idx], self.durations[idx]

class DeepHit(nn.Module):
    def __init__(self, input_dim, num_bins, hidden_dims=[64, 32], dropout=0.5):
        super(DeepHit, self).__init__()
        
        self.fc1 = nn.Linear(input_dim, hidden_dims[0])
        self.bn1 = nn.BatchNorm1d(hidden_dims[0])
        self.drop1 = nn.Dropout(dropout)
        
        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.bn2 = nn.BatchNorm1d(hidden_dims[1])
        self.drop2 = nn.Dropout(dropout)
        
        self.skip = nn.Linear(input_dim, hidden_dims[1])
        self.output = nn.Linear(hidden_dims[1], num_bins)
        
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
        
    def forward(self, x):
        h1 = self.drop1(torch.relu(self.bn1(self.fc1(x))))
        h2 = self.drop2(torch.relu(self.bn2(self.fc2(h1))))
        h2 = h2 + self.skip(x)
        return torch.softmax(self.output(h2), dim=1)


class DeepHitLoss(nn.Module):
    def __init__(self, alpha=0.5, sigma=0.05):
        super().__init__()
        self.alpha = alpha
        self.sigma = sigma

    def forward(self, pdf, y_bin, event, y_true):
        pdf = torch.clamp(pdf, min=1e-7, max=1.0 - 1e-7)
        cdf = torch.cumsum(pdf, dim=1)
        mask = torch.zeros_like(pdf).scatter(1, y_bin.view(-1, 1), 1)
        
        prob_event = (pdf * mask).sum(dim=1) + 1e-7
        prob_censored = 1.0 - (cdf * mask).sum(dim=1) + 1e-7
        nll = -torch.mean(event * torch.log(prob_event) + (1 - event) * torch.log(prob_censored))

        event_i = event.view(-1, 1)
        time_i = y_true.view(-1, 1)
        time_j = y_true.view(1, -1)
        A_ij = ((event_i == 1) & (time_i < time_j)).float()
        
        if A_ij.sum() < 1:
            return nll
        
        cdf_at_ti = (cdf * mask).sum(dim=1)
        diff = torch.clamp((cdf_at_ti.view(-1, 1) - cdf_at_ti.view(1, -1)) / self.sigma, -20, 20)
        rank = (A_ij * torch.exp(-diff)).sum() / (A_ij.sum() + 1e-8)
        
        return nll + self.alpha * rank


def train_model(X_tr, t_tr, e_tr, X_val, t_val, e_val, discretizer, config, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    t_tr_bin = discretizer.transform(t_tr)
    t_val_bin = discretizer.transform(t_val)
    
    train_loader = DataLoader(SurvivalDataset(X_tr, t_tr_bin, e_tr, t_tr), 
                              batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(SurvivalDataset(X_val, t_val_bin, e_val, t_val), 
                            batch_size=config['batch_size'], shuffle=False)
    
    model = DeepHit(X_tr.shape[1], discretizer.num_bins, 
                   config['hidden_dims'], config['dropout']).to(device)
    criterion = DeepHitLoss(config['alpha'], config['sigma'])
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'], 
                           weight_decay=config['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', 
                                                     factor=0.5, patience=10)
    
    best_c_index = 0
    best_state = None
    patience = 0
    history = {'train_loss': [], 'val_c_index': []}
    
    y_train_surv = Surv.from_arrays(event=e_tr.astype(bool), time=t_tr)
    y_val_surv = Surv.from_arrays(event=e_val.astype(bool), time=t_val)
    
    for epoch in range(config['epochs']):
        model.train()
        epoch_loss = 0
        for batch_X, batch_t_bin, batch_e, batch_t_raw in train_loader:
            batch_X = batch_X.to(device)
            batch_t_bin, batch_e, batch_t_raw = batch_t_bin.to(device), batch_e.to(device), batch_t_raw.to(device)
            
            optimizer.zero_grad()
            pdf = model(batch_X)
            loss = criterion(pdf, batch_t_bin, batch_e, batch_t_raw)
            
            if not torch.isnan(loss):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_loss += loss.item()
        
        model.eval()
        val_preds = []
        with torch.no_grad():
            for batch_X, _, _, _ in val_loader:
                pdf = model(batch_X.to(device))
                cdf = torch.cumsum(pdf, dim=1)
                risk = cdf[:, discretizer.num_bins // 2].cpu().numpy()
                val_preds.append(risk)
        
        val_preds = np.concatenate(val_preds)
        
        try:
            c_index = concordance_index_ipcw(y_train_surv, y_val_surv, val_preds, tau=None)[0]
        except:
            c_index = concordance_index_censored(e_val.astype(bool), t_val, val_preds)[0]
        
        history['train_loss'].append(epoch_loss / len(train_loader))
        history['val_c_index'].append(c_index)
        
        scheduler.step(c_index)
        
        if c_index > best_c_index:
            best_c_index = c_index
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= config['patience']:
                break
    
    model.load_state_dict(best_state)
    return model, best_c_index, history

def plot_training_curves(histories, config_names, save_path='./plots/'):
    import os
    os.makedirs(save_path, exist_ok=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for hist, name in zip(histories, config_names):
        axes[0].plot(hist['train_loss'], label=name, alpha=0.7)
        axes[1].plot(hist['val_c_index'], label=name, alpha=0.7)
    
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Training Loss')
    axes[0].set_title('Training Loss by Epoch')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('C-Index')
    axes[1].set_title('Validation C-Index by Epoch')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{save_path}deephit_training_curves.png', dpi=150, bbox_inches='tight')
    print(f"Training curves: {save_path}deephit_training_curves.png")
    plt.close()


def plot_risk_distribution(risk_scores, y_event, save_path='./plots/'):
    import os
    os.makedirs(save_path, exist_ok=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    events = risk_scores[y_event == 1]
    censored = risk_scores[y_event == 0]
    
    ax.hist(censored, bins=50, alpha=0.6, label='Censored', color='blue', density=True)
    ax.hist(events, bins=50, alpha=0.6, label='Events', color='red', density=True)
    
    ax.set_xlabel('Predicted Risk Score')
    ax.set_ylabel('Density')
    ax.set_title('Distribution of Predicted Risk Scores')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{save_path}deephit_risk_distribution.png', dpi=150, bbox_inches='tight')
    print(f"Risk distribution: {save_path}deephit_risk_distribution.png")
    plt.close()


def plot_kaplan_meier(y_time, y_event, risk_scores, n_groups=4, save_path='./plots/'):
    import os
    os.makedirs(save_path, exist_ok=True)
    
    risk_quantiles = pd.qcut(risk_scores, q=n_groups, labels=[f'Q{i+1}' for i in range(n_groups)])
    
    fig, ax = plt.subplots(figsize=(10, 6))
    kmf = KaplanMeierFitter()
    
    for group in risk_quantiles.unique():
        mask = risk_quantiles == group
        kmf.fit(y_time[mask], y_event[mask], label=f'Risk {group}')
        kmf.plot_survival_function(ax=ax, ci_show=True)
    
    ax.set_xlabel('Time (years)')
    ax.set_ylabel('Survival Probability')
    ax.set_title('Kaplan-Meier Curves by Risk Quartile')
    ax.grid(alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(f'{save_path}deephit_kaplan_meier.png', dpi=150, bbox_inches='tight')
    print(f"Kaplan-Meier: {save_path}deephit_kaplan_meier.png")
    plt.close()

df_train = pd.read_csv('./data/data_processed/df_ready_train.csv')
df_test = pd.read_csv('./data/data_processed/df_ready_test.csv')

df_train = df_train.dropna(subset=['OS_YEARS', 'OS_STATUS'])
print(f"Dataset: {len(df_train)} samples")

exclude = ['ID', 'OS_YEARS', 'OS_STATUS']
features = [c for c in df_train.columns if c not in exclude]

X = df_train[features].values.astype(np.float32)
y_time = df_train['OS_YEARS'].values.astype(np.float32)
y_event = df_train['OS_STATUS'].values.astype(int)
X_test = df_test[features].values.astype(np.float32)

print(f"Features: {len(features)}, Events: {y_event.sum()} ({y_event.mean()*100:.1f}%)")

X_tr, X_val, t_tr, t_val, e_tr, e_val = train_test_split(
    X, y_time, y_event, test_size=0.15, stratify=y_event, random_state=42
)

imputer = SimpleImputer(strategy='median')
X_tr = imputer.fit_transform(X_tr)
X_val = imputer.transform(X_val)
X_test = imputer.transform(X_test)

scaler = RobustScaler()
X_tr = scaler.fit_transform(X_tr)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

discretizer = TimeDiscretizer(num_bins=50)
discretizer.fit(t_tr, e_tr)
print(f"Discretization: {discretizer.num_bins} bins")

config = {
    'alpha': 0.5,
    'sigma': 0.05,
    'lr': 5e-4,
    'dropout': 0.5,
    'hidden_dims': [64, 32],
    'batch_size': 32,
    'epochs': 250,
    'patience': 25,
    'weight_decay': 3e-3
}


print("TRAINING")


models = []
scores = []
histories = []
test_preds = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for i in range(5):
    print(f"\nModel {i+1}/5")
    model, score, history = train_model(X_tr, t_tr, e_tr, X_val, t_val, e_val, 
                                       discretizer, config, seed=i*10+42)
    print(f"  C-Index: {score:.4f}")
    
    models.append(model)
    scores.append(score)
    histories.append(history)
    
    model.eval()
    with torch.no_grad():
        pdf = model(torch.FloatTensor(X_test).to(device))
        cdf = torch.cumsum(pdf, dim=1)
        risk = cdf[:, discretizer.num_bins // 2].cpu().numpy()
        test_preds.append(risk)

final_preds = np.mean(test_preds, axis=0)
avg_score = np.mean(scores)
std_score = np.std(scores)


print(f"Final Score: {avg_score:.4f} ± {std_score:.4f}")


submission = pd.DataFrame({
    'ID': df_test['ID'],
    'risk_score': final_preds
})
submission.to_csv(f'./submission/deephit_{avg_score:.4f}.csv', index=False)
print(f"Submission saved")

# Validation predictions for plots
val_preds_all = []
for model in models:
    model.eval()
    with torch.no_grad():
        pdf = model(torch.FloatTensor(X_val).to(device))
        cdf = torch.cumsum(pdf, dim=1)
        risk = cdf[:, discretizer.num_bins // 2].cpu().numpy()
        val_preds_all.append(risk)

val_preds_final = np.mean(val_preds_all, axis=0)

print("GENERATING PLOTS")

plot_training_curves(histories, [f'Model {i+1}' for i in range(5)])
plot_risk_distribution(val_preds_final, e_val)
plot_kaplan_meier(t_val, e_val, val_preds_final)


Preprocessing
17 features

GRID SEARCH INTELLIGENT
Total configurations: 243
Test rapide: 150 epochs, patience=35

[1/243] T=16 H=(80, 40) lr=8e-04 α=0.25 L2=5e-05 → 0.64310
[2/243] T=16 H=(80, 40) lr=8e-04 α=0.25 L2=1e-04 → 0.59034
[3/243] T=16 H=(80, 40) lr=8e-04 α=0.25 L2=2e-04 → 0.54236
[4/243] T=16 H=(80, 40) lr=8e-04 α=0.35 L2=5e-05 → 0.64383
[5/243] T=16 H=(80, 40) lr=8e-04 α=0.35 L2=1e-04 → 0.59315
[6/243] T=16 H=(80, 40) lr=8e-04 α=0.35 L2=2e-04 → 0.54309
[7/243] T=16 H=(80, 40) lr=8e-04 α=0.45 L2=5e-05 → 0.64251
[8/243] T=16 H=(80, 40) lr=8e-04 α=0.45 L2=1e-04 → 0.59510
[9/243] T=16 H=(80, 40) lr=8e-04 α=0.45 L2=2e-04 → 0.54346
[10/243] T=16 H=(80, 40) lr=1e-03 α=0.25 L2=5e-05 → 0.66080
[11/243] T=16 H=(80, 40) lr=1e-03 α=0.25 L2=1e-04 → 0.60782
[12/243] T=16 H=(80, 40) lr=1e-03 α=0.25 L2=2e-04 → 0.54835
[13/243] T=16 H=(80, 40) lr=1e-03 α=0.35 L2=5e-05 → 0.65926
[14/243] T=16 H=(80, 40) lr=1e-03 α=0.35 L2=1e-04 → 0.61224
[15/243] T=16 H=(80, 40) lr=1e-03 α=0.35 L2=2e-04 → 0.